# Solar Filament Segmentation Challenge 2026 - Solution Demo

This notebook demonstrates the complete end-to-end pipeline for the IEEE BigData Cup / MAGFiLO Solar Filament Segmentation Challenge 2026.

**Objective**: Instance-level segmentation of solar filaments from high-resolution (2048x2048) 8-bit grayscale H-Alpha solar imagery.

**Primary Metric**: Panoptic Quality (PQ) @ IoU threshold 0.5

In [ ]:
# Install dependencies
%pip install -q -r requirements.txt

In [ ]:
import os
import sys
import json
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

# Add src to path
sys.path.append('src')

from src.dataset import SolarFilamentDataset, get_dataloader, create_train_val_split
from src.augmentations import get_train_augmentations, get_val_augmentations
from src.models import FilamentUNet
from src.losses import TverskyFocalLoss
from src.postprocess import postprocess_masks
from src.evaluator import FilamentEvaluator
from src.rle_utils import encode_rle, to_submission_format

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Data Exploration

In [ ]:
# Load annotations
json_path = 'filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'
images_dir = 'filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images'

with open(json_path, 'r') as f:
    coco_data = json.load(f)

print("Dataset Info:")
print(f"  Images: {len(coco_data['images'])}")
print(f"  Annotations: {len(coco_data['annotations'])}")
print(f"  Categories: {coco_data['categories']}")

# Show first image info
print("\nFirst image:")
print(json.dumps(coco_data['images'][0], indent=2))

# Show first annotation
print("\nFirst annotation:")
print(json.dumps(coco_data['annotations'][0], indent=2))

In [ ]:
# Analyze annotator distribution
annotator_ids = set()
for img in coco_data['images']:
    annotator = img['id'].split('-')[0]
    annotator_ids.add(annotator)

print(f"Unique annotators: {len(annotator_ids)}")
print(f"Annotator IDs: {sorted(annotator_ids)}")

# Count images per annotator
from collections import Counter
annotator_counts = Counter()
for img in coco_data['images']:
    annotator = img['id'].split('-')[0]
    annotator_counts[annotator] += 1

for ann, count in sorted(annotator_counts.items()):
    print(f"  {ann}: {count} images")

## 2. Dataset Visualization

In [ ]:
# Create dataset with transforms
train_transform = get_train_augmentations()
val_transform = get_val_augmentations()

dataset = SolarFilamentDataset(
    json_path=json_path,
    images_dir=images_dir,
    transform=val_transform,  # Use val transform for visualization
    mode='val',
    apply_clahe=True,
    apply_limb_mask=True,
)

print(f"Dataset size: {len(dataset)}")

# Visualize a few samples
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i in range(3):
    idx = i * 100  # Sample different images
    sample = dataset[idx]
    
    image = sample['image'].permute(1, 2, 0).numpy()
    # Denormalize
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = image * std + mean
    image = np.clip(image, 0, 1)
    
    target = sample['target']
    masks = target['masks'].numpy()
    orig_id = target['orig_image_id']
    
    # Original image (grayscale)
    axes[i, 0].imshow(image[:, :, 0], cmap='gray')
    axes[i, 0].set_title(f"Image: {orig_id}")
    axes[i, 0].axis('off')
    
    # Combined mask
    if len(masks) > 0:
        combined = masks.sum(axis=0)
        combined = np.clip(combined, 0, 1)
    else:
        combined = np.zeros_like(image[:, :, 0])
    axes[i, 1].imshow(combined, cmap='hot')
    axes[i, 1].set_title(f"GT Masks ({len(masks)} instances)")
    axes[i, 1].axis('off')
    
    # Individual instances
    for j in range(min(2, len(masks))):
        axes[i, j+2].imshow(masks[j], cmap='gray')
        axes[i, j+2].set_title(f"Instance {j+1} (area: {masks[j].sum()}) ")
        axes[i, j+2].axis('off')
    
    for j in range(len(masks), 2):
        axes[i, j+2].axis('off')

plt.tight_layout()
plt.show()

## 3. Solar Limb Masking & CLAHE Demo

In [ ]:
# Demonstrate limb masking and CLAHE
sample_idx = 0
img_info = dataset.image_infos[sample_idx]
file_name = img_info['file_name']
img_path = os.path.join(images_dir, file_name)

original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

# Apply limb mask
masked = dataset._apply_limb_mask(original)

# Apply CLAHE
clahe = dataset._apply_clahe(masked)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(original, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(masked, cmap='gray')
axes[1].set_title('After Limb Mask')
axes[1].axis('off')

axes[2].imshow(clahe, cmap='gray')
axes[2].set_title('After CLAHE')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"Original range: [{original.min()}, {original.max()}]")
print(f"Masked range: [{masked.min()}, {masked.max()}]")
print(f"CLAHE range: [{clahe.min()}, {clahe.max()}]")

## 4. Model Training Setup

In [ ]:
# Create train/val split
train_infos, val_infos = create_train_val_split(json_path, val_ratio=0.1, seed=42)
print(f"Train samples: {len(train_infos)}")
print(f"Val samples: {len(val_infos)}")

# Create datasets
train_dataset = SolarFilamentDataset(
    json_path=json_path,
    images_dir=images_dir,
    transform=train_transform,
    mode='train',
)
train_dataset.image_infos = train_infos

val_dataset = SolarFilamentDataset(
    json_path=json_path,
    images_dir=images_dir,
    transform=val_transform,
    mode='val',
)
val_dataset.image_infos = val_infos

# Create dataloaders
train_loader = get_dataloader(
    json_path, images_dir,
    batch_size=2,
    num_workers=2,
    mode='train',
)
train_loader.dataset.image_infos = train_infos

val_loader = get_dataloader(
    json_path, images_dir,
    batch_size=2,
    num_workers=2,
    mode='val',
)
val_loader.dataset.image_infos = val_infos

In [ ]:
# Build model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FilamentUNet(encoder='resnet34', num_classes=1, num_instances=32)
model.to(device)

# Loss and optimizer
loss_fn = TverskyFocalLoss(
    tversky_alpha=0.3,
    tversky_beta=0.7,
    focal_alpha=0.25,
    focal_gamma=2.0,
    tversky_weight=0.7,
    focal_weight=0.3,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test forward pass
model.eval()
with torch.no_grad():
    images, targets = next(iter(val_loader))
    images = images.to(device)
    
    instance_logits, semantic_logits = model(images)
    
    print(f"Input shape: {images.shape}")
    print(f"Instance logits shape: {instance_logits.shape}")
    print(f"Semantic logits shape: {semantic_logits.shape}")
    
    # Get predictions
    instance_probs = torch.sigmoid(instance_logits)
    semantic_probs = torch.sigmoid(semantic_logits)
    combined = instance_probs * semantic_probs
    
    print(f"Combined shape: {combined.shape}")
    print(f"Combined range: [{combined.min():.4f}, {combined.max():.4f}]")

## 5. Postprocessing Demo

In [ ]:
# Demonstrate postprocessing on sample predictions
model.eval()
with torch.no_grad():
    images, targets = next(iter(val_loader))
    images = images.to(device)
    
    instance_logits, semantic_logits = model(images)
    instance_probs = torch.sigmoid(instance_logits)
    semantic_probs = torch.sigmoid(semantic_logits)
    combined = instance_probs * semantic_probs
    
    # Process first image in batch
    pred_masks = combined[0].cpu().numpy()  # (num_instances, H, W)
    pred_masks = (pred_masks > 0.5).astype(np.uint8)
    pred_scores = pred_masks.mean(axis=(1, 2))
    
    # Filter by score
    keep = pred_scores > 0.1
    pred_masks = pred_masks[keep]
    pred_scores = pred_scores[keep]
    
    print(f"Raw predictions: {len(pred_masks)} instances")
    
    # Get original image for barb reconnection
    orig_image = images[0].permute(1, 2, 0).cpu().numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    orig_image = orig_image * std + mean
    orig_image = np.clip(orig_image, 0, 1)
    orig_image = (orig_image * 255).astype(np.uint8)
    
    # Postprocess
    postprocess_config = {
        'nms_iou_threshold': 0.4,
        'min_size': 150,
        'watershed_min_distance': 10,
        'watershed_threshold_rel': 0.3,
        'merge_iou_threshold': 0.7,
        'barb_reconnect': True,
        'barb_max_gap': 5,
    }
    
    processed_masks, processed_scores = postprocess_masks(
        pred_masks, pred_scores, orig_image, postprocess_config
    )
    
    print(f"After postprocessing: {len(processed_masks)} instances")

In [ ]:
# Visualize postprocessing results
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Original image
axes[0, 0].imshow(orig_image[:, :, 0], cmap='gray')
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

# Ground truth
gt_masks = targets[0]['masks'].numpy()
if len(gt_masks) > 0:
    gt_combined = gt_masks.sum(axis=0)
    gt_combined = np.clip(gt_combined, 0, 1)
else:
    gt_combined = np.zeros_like(orig_image[:, :, 0])
axes[0, 1].imshow(gt_combined, cmap='hot')
axes[0, 1].set_title(f'Ground Truth ({len(gt_masks)} instances)')
axes[0, 1].axis('off')

# Raw predictions
if len(pred_masks) > 0:
    raw_combined = pred_masks.sum(axis=0)
    raw_combined = np.clip(raw_combined, 0, 1)
else:
    raw_combined = np.zeros_like(orig_image[:, :, 0])
axes[0, 2].imshow(raw_combined, cmap='hot')
axes[0, 2].set_title(f'Raw Predictions ({len(pred_masks)} instances)')
axes[0, 2].axis('off')

# Processed predictions
if len(processed_masks) > 0:
    proc_combined = processed_masks.sum(axis=0)
    proc_combined = np.clip(proc_combined, 0, 1)
else:
    proc_combined = np.zeros_like(orig_image[:, :, 0])
axes[0, 3].imshow(proc_combined, cmap='hot')
axes[0, 3].set_title(f'Postprocessed ({len(processed_masks)} instances)')
axes[0, 3].axis('off')

# Individual processed instances
for j in range(min(3, len(processed_masks))):
    row = 1
    col = j
    axes[row, col].imshow(processed_masks[j], cmap='gray')
    axes[row, col].set_title(f'Instance {j+1} (score: {processed_scores[j]:.3f})')
    axes[row, col].axis('off')

for j in range(len(processed_masks), 3):
    axes[1, j].axis('off')

axes[1, 3].axis('off')

plt.tight_layout()
plt.show()

## 6. Evaluation Demo

In [ ]:
# Evaluate on validation set
evaluator = FilamentEvaluator(iou_threshold=0.5)

all_predictions = []
all_ground_truths = []

model.eval()
with torch.no_grad():
    for images, targets in tqdm(val_loader, desc='Evaluating'):
        images = images.to(device)
        
        instance_logits, semantic_logits = model(images)
        instance_probs = torch.sigmoid(instance_logits)
        semantic_probs = torch.sigmoid(semantic_logits)
        combined = instance_probs * semantic_probs
        
        for b in range(images.shape[0]):
            pred_masks = combined[b].cpu().numpy()
            pred_masks = (pred_masks > 0.5).astype(np.uint8)
            pred_scores = pred_masks.mean(axis=(1, 2))
            
            keep = pred_scores > 0.1
            pred_masks = pred_masks[keep]
            
            # Postprocess
            orig_image = images[b].permute(1, 2, 0).cpu().numpy()
            orig_image = orig_image * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            orig_image = np.clip(orig_image, 0, 1)
            orig_image = (orig_image * 255).astype(np.uint8)
            
            processed_masks, _ = postprocess_masks(
                pred_masks, pred_scores[keep], orig_image, postprocess_config
            )
            
            all_predictions.append(processed_masks)
            
            # Ground truth
            gt_masks = targets[b]['masks'].numpy()
            gt_list = [gt_masks[i] for i in range(gt_masks.shape[0]) if gt_masks[i].sum() >= 150]
            all_ground_truths.append(gt_list)

# Compute metrics
metrics = evaluator.evaluate_dataset(all_predictions, all_ground_truths)
evaluator.print_summary(metrics)

In [ ]:
# Plot diagnostic distributions
evaluator.plot_diagnostics()

## 7. RLE Encoding & Submission Format

In [ ]:
# Demonstrate RLE encoding
# Create a sample mask
sample_mask = np.zeros((2048, 2048), dtype=np.uint8)
sample_mask[500:600, 500:600] = 1
sample_mask[1000:1100, 1000:1100] = 1

# Encode to RLE
rle = encode_rle(sample_mask)
print(f"RLE size: {rle['size']}")
print(f"RLE counts (first 100 chars): {rle['counts'][:100]}...")

# Decode back
decoded = decode_rle(rle)
print(f"Decoded shape: {decoded.shape}")
print(f"Decoded sum: {decoded.sum()}")
print(f"Match: {np.array_equal(sample_mask, decoded)}")

# Submission format
filament_id, seg_rle = to_submission_format(rle, '010401-20160920230134Lh', 0)
print(f"\nFilament ID: {filament_id}")
print(f"Segmentation RLE (first 50 chars): {seg_rle[:50]}...")

## 8. Full Training Run (Example)

In [ ]:
# Example training loop (run for a few epochs to verify)
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    
    for images, targets in loader:
        images = images.to(device)
        
        # Prepare target masks
        batch_masks = []
        for target in targets:
            masks = target['masks']
            if masks.shape[0] > 0:
                combined = masks.sum(dim=0).clamp(0, 1)
            else:
                combined = torch.zeros_like(masks[0:1]).squeeze(0)
            batch_masks.append(combined)
        
        target_masks = torch.stack(batch_masks).to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        instance_logits, semantic_logits = model(images)
        loss = loss_fn(instance_logits, target_masks)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

# Run a few epochs
print("Running mini-training (3 epochs)...")
for epoch in range(3):
    loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    scheduler.step()
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}")

## 9. Test Inference & Submission Generation

In [ ]:
# End-to-end submission generation
!python submission_pipeline.py --model_path checkpoints/best_model.pth --model_type unet --device cuda

# Validate the exported RLE format
from submission_pipeline import validate_submission_format
validate_submission_format('submission.csv')